# 03 — RAG Evaluation
**PaperSloth — Retrieval Quality + Faithfulness + Ablation Study**

```
Evaluation plan:
  1. Setup & connections
  2. Auto-generate eval dataset from ingested parent chunks (Gemini)
  3. Review + add manual negatives
  4. Precision@5 — hybrid vs dense-only (ablation)
  5. RAGAS — Faithfulness + Answer Relevancy (no ground truth needed)
  6. Results summary + export
```

**What we're evaluating:**
- PaperSloth is a *question retrieval* RAG system — documents are exam questions, not answers
- Retrieval quality (Precision@5): did we surface the right questions for a topic query?
- Faithfulness: does the LLM response only contain info grounded in retrieved context?
- Answer Relevancy: does the response actually address what the student asked?
- Ablation: hybrid (alpha=0.7) vs dense-only (alpha=1.0) to validate architecture decision

**Note on labels:**  
With 45+ papers ingested, `question_number` is ambiguous across semesters (Q2 in May 2024 ≠ Q2 in September 2025).  
We use `parent_id` as the ground truth label instead — it is unique per question per paper.

---
## Section 1 — Setup & Connections

In [1]:
import os, json, pickle, time, statistics
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

keys = ["GEMINI_API_KEY", "PINECONE_API_KEY", "DATABASE_URL"]
for k in keys:
    print(f"{k}: {'✅' if os.getenv(k) else '❌ MISSING'}")

GEMINI_API_KEY: ✅
PINECONE_API_KEY: ✅
DATABASE_URL: ✅


In [2]:
import ollama
import psycopg2
import google.generativeai as genai
from pinecone import Pinecone
from sentence_transformers import CrossEncoder

# ── Clients ───────────────────────────────────────────────────────────────────
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("papersloth")

# ── Models ────────────────────────────────────────────────────────────────────
EMBED_MODEL   = "nomic-embed-text-v2-moe:latest"
GEMINI_MODEL  = "gemma-4-31b-it"
RERANKER_PATH = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BM25_PATH     = "data/bm25_model.pkl"

# ── Load BM25 ─────────────────────────────────────────────────────────────────
assert Path(BM25_PATH).exists(), f"BM25 not found at {BM25_PATH}"
with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

# ── Load reranker ─────────────────────────────────────────────────────────────
reranker = CrossEncoder(RERANKER_PATH)

# ── Gemini generation model (for RAG answers) ─────────────────────────────────
RAG_SYSTEM = (
    "You are a UTP past year exam assistant. "
    "You will be given exam questions and a student query. "
    "Output ONLY the matching question text with its marks and source. "
    "Do not explain, reason, or analyse. "
    "Begin your response immediately with the answer."
)
gen_model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=RAG_SYSTEM)

# ── Gemini query generation model (for eval dataset) ─────────────────────────
query_gen_model = genai.GenerativeModel("gemini-3.1-flash-lite")

# ── Pinecone stats ────────────────────────────────────────────────────────────
stats = index.describe_index_stats()
print(f"✅ All clients ready")
print(f"   Pinecone: {stats['total_vector_count']} vectors")

/var/folders/my/2fkdjjvj7h9dn3nmk1ttlccc0000gn/T/ipykernel_28140/3437348188.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ All clients ready
   Pinecone: 158 vectors


---
## Section 2 — Core Retrieval Helpers

Extracted from `services/retrieval/` so this notebook is self-contained.

In [3]:
def embed_query(text: str) -> list[float]:
    resp = ollama.embed(model=EMBED_MODEL, input=f"search_query: {text}")
    return resp["embeddings"][0]


def hybrid_scale(dense, sparse, alpha=0.7):
    return (
        [v * alpha for v in dense],
        {"indices": sparse["indices"], "values": [v * (1 - alpha) for v in sparse["values"]]},
    )


def fetch_parents(parent_ids: list[str]) -> list[dict]:
    if not parent_ids:
        return []
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur  = conn.cursor()
    cur.execute("""
        SELECT parent_id, question_number, full_text, total_marks,
               course_code, semester, year
        FROM   parent_chunks
        WHERE  parent_id = ANY(%s)
    """, (parent_ids,))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return [
        {
            "parent_id":       r[0],
            "question_number": r[1],
            "full_text":       r[2],
            "total_marks":     r[3],
            "course_code":     r[4],
            "semester":        r[5],
            "year":            r[6],
        }
        for r in rows
    ]


def retrieve(
    query: str,
    course_code: str = None,
    top_k: int = 20,
    rerank_top_n: int = 5,
    alpha: float = 0.7,
) -> list[dict]:
    """
    Full retrieval pipeline returning top parent chunks.
    alpha=0.7 → hybrid (production), alpha=1.0 → dense-only (baseline).
    """
    dense  = embed_query(query)
    sparse = bm25.encode_queries([query])[0]
    d, s   = hybrid_scale(dense, sparse, alpha)

    pinecone_filter = {"course_code": {"$eq": course_code}} if course_code else None

    results = index.query(
        vector           = d,
        sparse_vector    = s,
        top_k            = top_k,
        include_metadata = True,
        filter           = pinecone_filter,
    )

    if not results.matches:
        return []

    passages = [m.metadata.get("text_preview", "") for m in results.matches]
    scores   = reranker.predict([(query, p) for p in passages])
    ranked   = sorted(zip(results.matches, scores), key=lambda x: x[1], reverse=True)
    top      = [m for m, _ in ranked[:rerank_top_n]]

    parent_ids = list({m.metadata["parent_id"] for m in top})
    return fetch_parents(parent_ids)


def generate_answer(query: str, parents: list[dict]) -> str:
    """LLM generation over retrieved context — used for RAGAS metrics."""
    if not parents:
        return "No relevant questions found."
    parts = []
    for doc in parents:
        header = (
            f"[{doc['course_code']} | {doc['semester']} {doc['year']} | "
            f"Q{doc['question_number']} | {doc['total_marks']} marks]"
        )
        parts.append(f"{header}\n{doc['full_text']}")
    context = "\n\n---\n\n".join(parts)
    prompt  = f"EXAM QUESTIONS:\n{context}\n\nSTUDENT: {query}"
    resp = gen_model.generate_content(prompt, generation_config={"temperature": 0.1})
    return resp.text


print("✅ Retrieval helpers ready")

✅ Retrieval helpers ready


---
## Section 3 — Auto-Generate Eval Dataset

With 45 papers ingested, writing queries manually is too slow.

**Strategy:**
1. Pull all parent chunks from PostgreSQL
2. For each chunk, ask Gemini to generate one natural student search query
3. Label is the `parent_id` of that chunk — guaranteed relevant
4. Review the output, remove any bad queries
5. Add a few manual negative queries at the end

**Why `parent_id` labels instead of `question_number`:**  
With 45 papers, `question_number = "2"` appears in every paper.  
`parent_id` is unique (e.g. `RBB3013_2025_September2025__Q2`) so there is no ambiguity.

In [4]:
# ── 3.1 Pull all parent chunks from PostgreSQL ────────────────────────────────

conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur  = conn.cursor()
cur.execute("""
    SELECT parent_id, question_number, course_code, semester, year, full_text
    FROM   parent_chunks
    ORDER  BY course_code, year, semester, question_number;
""")
all_chunks = cur.fetchall()
cur.close(); conn.close()

print(f"✅ Fetched {len(all_chunks)} parent chunks")
print()

# Preview breakdown by course
from collections import Counter
course_counts = Counter(row[2] for row in all_chunks)
for course, count in sorted(course_counts.items()):
    print(f"  {course}: {count} questions")

✅ Fetched 45 parent chunks

  EDB2613: 25 questions
  RBB3013: 20 questions


In [5]:
# ── 3.2 Define query generation function ─────────────────────────────────────

QUERY_GEN_PROMPT = """Given this university exam question, generate ONE natural search query \
that a student would type to find this question when studying.

Rules:
- The query should be 4-10 words
- Focus on the KEY TOPIC or CONCEPT being tested, not the question format
- Do NOT include course codes, question numbers, or semester info in the query
- Use vocabulary a student would naturally use (not overly formal)
- Return ONLY a valid JSON object with a single key "query". No markdown, no explanation.

Exam question:
{full_text}"""


def generate_eval_query(full_text: str) -> str | None:
    """
    Ask Gemini to generate a natural student search query for a given exam question.
    Returns the query string, or None if generation failed.
    """
    prompt = QUERY_GEN_PROMPT.format(full_text=full_text[:600])  # truncate to save tokens
    try:
        resp = query_gen_model.generate_content(
            prompt,
            generation_config={"temperature": 0.4}
        )
        text = resp.text.strip().replace("```json", "").replace("```", "").strip()
        return json.loads(text)["query"]
    except Exception as e:
        return None


# Quick sanity check on one chunk
test_chunk = all_chunks[0]
test_query = generate_eval_query(test_chunk[5])
print(f"parent_id : {test_chunk[0]}")
print(f"generated : {test_query}")

parent_id : EDB2613_2022_MAY2022SEMESTER__Q1
generated : how to find transfer function poles and zeros


In [6]:
# ── 3.3 Batch generate queries for all parent chunks ─────────────────────────
#
# Saves progress to data/eval_dataset_generated.json as it goes.
# If interrupted, re-running will skip already-generated entries.
#
# Rate limit: gemini-2.0-flash allows ~15 RPM on free tier → 0.5s sleep is safe.
# On paid tier you can reduce to 0.1s.

SAVE_PATH = Path("data/eval_dataset_generated.json")
SAVE_PATH.parent.mkdir(exist_ok=True)

# Load existing progress if resuming
if SAVE_PATH.exists():
    with open(SAVE_PATH) as f:
        eval_dataset = json.load(f)
    done_ids = {item["relevant_parent_ids"][0] for item in eval_dataset}
    print(f"Resuming — {len(eval_dataset)} already done, {len(all_chunks) - len(done_ids)} remaining")
else:
    eval_dataset = []
    done_ids     = set()
    print(f"Starting fresh — {len(all_chunks)} chunks to process")

print()

failed = []

for i, row in enumerate(all_chunks):
    parent_id, qnum, course, semester, year, full_text = row

    if parent_id in done_ids:
        continue

    query = generate_eval_query(full_text)

    if query is None:
        print(f"[{i+1:03d}] ⚠️  FAILED  {parent_id}")
        failed.append(parent_id)
        time.sleep(1)
        continue

    eval_dataset.append({
        "query":               query,
        "course_code":         course,
        "relevant_parent_ids": [parent_id],
        "query_type":          "topic_search",
    })
    done_ids.add(parent_id)

    print(f"[{i+1:03d}/{len(all_chunks)}] {parent_id}")
    print(f"          → {query}")

    # Save progress every 10 entries
    if len(eval_dataset) % 10 == 0:
        with open(SAVE_PATH, "w") as f:
            json.dump(eval_dataset, f, indent=2)

    time.sleep(0.5)

# Final save
with open(SAVE_PATH, "w") as f:
    json.dump(eval_dataset, f, indent=2)

print(f"\n✅ Done — {len(eval_dataset)} queries saved to {SAVE_PATH}")
if failed:
    print(f"⚠️  {len(failed)} failed: {failed}")

Resuming — 45 already done, 0 remaining


✅ Done — 45 queries saved to data/eval_dataset_generated.json


---
## Section 4 — Review & Finalise Eval Dataset

Before running evaluation, do a quick review:
- Skim the generated queries — remove any that are vague, wrong, or don't reflect the question
- Add manual negative queries (topics definitely NOT in any of your papers)

You can edit `data/eval_dataset_generated.json` directly, or use the cells below.

In [7]:
# ── 4.1 Load and preview generated dataset ───────────────────────────────────

with open("data/eval_dataset_generated.json") as f:
    eval_dataset = json.load(f)

print(f"Total generated queries: {len(eval_dataset)}")
print()

# Print all for review — skim for obviously bad ones
for i, item in enumerate(eval_dataset):
    pid = item["relevant_parent_ids"][0]
    print(f"[{i:03d}] {pid}")
    print(f"       {item['query']}")

Total generated queries: 45

[000] EDB2613_2022_MAY2022SEMESTER__Q1
       how to find transfer function poles and zeros
[001] EDB2613_2022_MAY2022SEMESTER__Q2
       how to design transfer function from step response parameters
[002] EDB2613_2022_MAY2022SEMESTER__Q3
       how to calculate steady state error for control systems
[003] EDB2613_2022_MAY2022SEMESTER__Q4
       how to evaluate transmitter accuracy and precision
[004] EDB2613_2022_MAY2022SEMESTER__Q5
       how to calibrate differential pressure level transmitter
[005] EDB2613_2022_September2022__Q1
       how to calculate second order system parameters from step response
[006] EDB2613_2022_September2022__Q2
       how to sketch root locus for lead compensator
[007] EDB2613_2022_September2022__Q3
       how to select a thermocouple for high temperature measurement
[008] EDB2613_2022_September2022__Q4
       how to calculate excitation voltage for wheatstone bridge strain gauge
[009] EDB2613_2023_JANUARY2023SEMESTER__Q1
    

In [8]:
# ── 4.2 Remove bad queries (optional) ────────────────────────────────────────
#
# Set BAD_INDICES to the indices of queries you want to drop (from the preview above).
# Leave empty if everything looks fine.

BAD_INDICES = set([
    # e.g. 12, 27, 45
])

if BAD_INDICES:
    before = len(eval_dataset)
    eval_dataset = [item for i, item in enumerate(eval_dataset) if i not in BAD_INDICES]
    print(f"Removed {before - len(eval_dataset)} bad queries → {len(eval_dataset)} remaining")
else:
    print("No queries removed")

No queries removed


In [9]:
# ── 4.3 Add manual negative queries ──────────────────────────────────────────
#
# Negative queries = topics that are definitely NOT in any of your papers.
# These test that your system doesn't hallucinate relevance.
# Aim for 5-10 negatives. relevant_parent_ids = [] means nothing should be retrieved.
#
# ⚠️  Be honest — only mark as negative if you're sure the topic isn't covered.

MANUAL_NEGATIVES = [
    {
        "query": "neural network backpropagation gradient descent",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "SQL database foreign key join query",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "organic chemistry reaction mechanism",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "project management agile scrum sprint",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "financial accounting balance sheet depreciation",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "principal component analysis for dataset variance reduction",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "rest api endpoint routing with fastapi and mongodb",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "reinforced concrete beam shear stress and bending moment",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "mobile application state management using jetpack compose",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    },
    {
        "query": "ad network traffic optimization and cookie tracking",
        "course_code": None,
        "relevant_parent_ids": [],
        "query_type": "negative",
    }
]

eval_dataset.extend(MANUAL_NEGATIVES)

# Save finalised dataset
FINAL_PATH = Path("data/eval_dataset_final.json")
with open(FINAL_PATH, "w") as f:
    json.dump(eval_dataset, f, indent=2)

topic_count    = sum(1 for q in eval_dataset if q["query_type"] == "topic_search")
negative_count = sum(1 for q in eval_dataset if q["query_type"] == "negative")

print(f"✅ Final eval dataset saved → {FINAL_PATH}")
print(f"   topic_search : {topic_count}")
print(f"   negative     : {negative_count}")
print(f"   total        : {len(eval_dataset)}")

✅ Final eval dataset saved → data/eval_dataset_final.json
   topic_search : 45
   negative     : 10
   total        : 55


---
## Section 5 — Precision@5 Ablation

Run each query twice:
- **Hybrid** (alpha=0.7): your production system
- **Dense-only** (alpha=1.0): baseline, no BM25

**Precision@K** = (relevant retrieved in top K) / K

A retrieved parent chunk is *relevant* if its `parent_id` is in `relevant_parent_ids`.

**For negative queries:** precision = 1.0 if nothing relevant retrieved, 0.0 if something is.

In [23]:
K = 5   

def precision_at_k(retrieved: list[dict], relevant_parent_ids: list[str], k: int = 5) -> float:
    """
    What fraction of the top-K retrieved questions are relevant?
    relevant_parent_ids: list of exact parent_id strings
    retrieved: list of parent dicts from fetch_parents()
    """
    if not relevant_parent_ids:
        # Negative query — correct if nothing retrieved
        return 1.0 if len(retrieved) == 0 else 0.0

    top_k = retrieved[:k]
    hits  = sum(1 for r in top_k if r["parent_id"] in relevant_parent_ids)
    return hits / min(k, max(len(top_k), 1))


print(f"✅ precision_at_k defined (K={K})")

✅ precision_at_k defined (K=5)


In [24]:
# ── Load finalised dataset ────────────────────────────────────────────────────
with open("data/eval_dataset_final.json") as f:
    eval_dataset = json.load(f)

print(f"Loaded {len(eval_dataset)} eval queries")

Loaded 55 eval queries


In [26]:
# ── Run ablation ──────────────────────────────────────────────────────────────
# Each query hits Pinecone + reranker twice (hybrid + dense).
# ~45 papers × avg 4 questions = ~180 queries × ~4s each ≈ 12 min total.

ablation_results = []

for i, item in enumerate(eval_dataset):
    query  = item["query"]
    course = item["course_code"]
    rel    = item["relevant_parent_ids"]
    qtype  = item["query_type"]

    print(f"[{i+1:03d}/{len(eval_dataset)}] {query[:60]}")

    # Hybrid (production)
    t0          = time.time()
    hybrid_ret  = retrieve(query, course_code=course, rerank_top_n=K, alpha=0.7)
    hybrid_time = time.time() - t0
    hybrid_p    = precision_at_k(hybrid_ret, rel, k=K)

    # Dense-only (baseline)
    t0         = time.time()
    dense_ret  = retrieve(query, course_code=course, rerank_top_n=K, alpha=1.0)
    dense_time = time.time() - t0
    dense_p    = precision_at_k(dense_ret, rel, k=K)

    ablation_results.append({
        "query":              query,
        "course_code":        course,
        "query_type":         qtype,
        "relevant_pids":      rel,
        "hybrid_precision":   hybrid_p,
        "dense_precision":    dense_p,
        "hybrid_retrieved":   [r["parent_id"] for r in hybrid_ret],
        "dense_retrieved":    [r["parent_id"] for r in dense_ret],
        "hybrid_time_s":      round(hybrid_time, 2),
        "dense_time_s":       round(dense_time, 2),
    })

    delta  = hybrid_p - dense_p
    symbol = "✅" if delta >= 0 else "⚠️ "
    print(f"         hybrid={hybrid_p:.2f}  dense={dense_p:.2f}  Δ={delta:+.2f} {symbol}")

print("\n✅ Ablation complete")

[001/55] how to find transfer function poles and zeros
         hybrid=0.33  dense=0.33  Δ=+0.00 ✅
[002/55] how to design transfer function from step response parameter
         hybrid=0.50  dense=0.50  Δ=+0.00 ✅
[003/55] how to calculate steady state error for control systems
         hybrid=0.33  dense=0.33  Δ=+0.00 ✅
[004/55] how to evaluate transmitter accuracy and precision
         hybrid=0.33  dense=0.33  Δ=+0.00 ✅
[005/55] how to calibrate differential pressure level transmitter
         hybrid=0.25  dense=0.25  Δ=+0.00 ✅
[006/55] how to calculate second order system parameters from step re
         hybrid=0.50  dense=0.50  Δ=+0.00 ✅
[007/55] how to sketch root locus for lead compensator
         hybrid=0.33  dense=0.33  Δ=+0.00 ✅
[008/55] how to select a thermocouple for high temperature measuremen
         hybrid=0.50  dense=0.50  Δ=+0.00 ✅
[009/55] how to calculate excitation voltage for wheatstone bridge st
         hybrid=0.33  dense=0.33  Δ=+0.00 ✅
[010/55] working princi

In [27]:
# ── Aggregate results ─────────────────────────────────────────────────────────

hybrid_scores = [r["hybrid_precision"] for r in ablation_results]
dense_scores  = [r["dense_precision"]  for r in ablation_results]

mean_hybrid = statistics.mean(hybrid_scores)
mean_dense  = statistics.mean(dense_scores)
improvement = (mean_hybrid - mean_dense) / max(mean_dense, 1e-9) * 100

topic_results    = [r for r in ablation_results if r["query_type"] == "topic_search"]
negative_results = [r for r in ablation_results if r["query_type"] == "negative"]

print("=" * 55)
print("PRECISION@5 RESULTS")
print("=" * 55)
print(f"  Hybrid (alpha=0.7)  : {mean_hybrid:.3f}")
print(f"  Dense-only (α=1.0)  : {mean_dense:.3f}")
print(f"  Improvement         : {improvement:+.1f}%")
print()
print(f"  Topic queries ({len(topic_results)})")
if topic_results:
    th = statistics.mean(r["hybrid_precision"] for r in topic_results)
    td = statistics.mean(r["dense_precision"]  for r in topic_results)
    print(f"    Hybrid : {th:.3f}")
    print(f"    Dense  : {td:.3f}")
print()
print(f"  Negative queries ({len(negative_results)})")
if negative_results:
    nh = statistics.mean(r["hybrid_precision"] for r in negative_results)
    nd = statistics.mean(r["dense_precision"]  for r in negative_results)
    print(f"    Hybrid : {nh:.3f}  (1.0 = correctly returned nothing)")
    print(f"    Dense  : {nd:.3f}")
print("=" * 55)

PRECISION@5 RESULTS
  Hybrid (alpha=0.7)  : 0.241
  Dense-only (α=1.0)  : 0.237
  Improvement         : +1.9%

  Topic queries (45)
    Hybrid : 0.295
    Dense  : 0.289

  Negative queries (10)
    Hybrid : 0.000  (1.0 = correctly returned nothing)
    Dense  : 0.000


In [28]:
# ── Per-query breakdown table ─────────────────────────────────────────────────
print(f"{'#':<4} {'Query':<50} {'Type':<14} {'Hybrid':>7} {'Dense':>7} {'Δ':>6}")
print("-" * 90)
for i, r in enumerate(ablation_results):
    delta = r["hybrid_precision"] - r["dense_precision"]
    flag  = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(
        f"{i+1:<4} {r['query'][:49]:<50} {r['query_type']:<14} "
        f"{r['hybrid_precision']:>7.2f} {r['dense_precision']:>7.2f} "
        f"{delta:>+5.2f}{flag}"
    )

#    Query                                              Type            Hybrid   Dense      Δ
------------------------------------------------------------------------------------------
1    how to find transfer function poles and zeros      topic_search      0.33    0.33 +0.00=
2    how to design transfer function from step respons  topic_search      0.50    0.50 +0.00=
3    how to calculate steady state error for control s  topic_search      0.33    0.33 +0.00=
4    how to evaluate transmitter accuracy and precisio  topic_search      0.33    0.33 +0.00=
5    how to calibrate differential pressure level tran  topic_search      0.25    0.25 +0.00=
6    how to calculate second order system parameters f  topic_search      0.50    0.50 +0.00=
7    how to sketch root locus for lead compensator      topic_search      0.33    0.33 +0.00=
8    how to select a thermocouple for high temperature  topic_search      0.50    0.50 +0.00=
9    how to calculate excitation voltage for wheatston  topic_s

---
## Section 6 — RAGAS: Faithfulness + Answer Relevancy

These metrics don't need ground truth answers — they evaluate the LLM generation layer.

- **Faithfulness**: is the generated answer grounded in the retrieved context?
  - High → LLM only states what is in the retrieved questions
  - Low → LLM is hallucinating question text or details

- **Answer Relevancy**: does the answer actually respond to what was asked?
  - High → student asked about thermocouples, answer is about thermocouples
  - Low → answer drifts off topic

We run this on topic_search queries only — negatives have no context to evaluate against.

**To keep costs down:** we sample up to 40 queries for RAGAS instead of running all of them.

In [10]:
# ── 6.-1 Re-init dependencies from Sections 1-2 (run after any kernel restart) ──
import os, json, pickle, time, statistics
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import ollama
import psycopg2
import google.generativeai as genai
from pinecone import Pinecone
from sentence_transformers import CrossEncoder

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("papersloth")

EMBED_MODEL   = "nomic-embed-text-v2-moe:latest"
GEMINI_MODEL  = "gemma-4-31b-it"
RERANKER_PATH = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BM25_PATH     = "data/bm25_model.pkl"

with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

reranker = CrossEncoder(RERANKER_PATH)

RAG_SYSTEM = (
    "You are a UTP past year exam assistant. "
    "You will be given exam questions and a student query. "
    "Output ONLY the matching question text with its marks and source. "
    "Do not explain, reason, or analyse. "
    "Begin your response immediately with the answer."
)
gen_model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=RAG_SYSTEM)


def embed_query(text: str) -> list[float]:
    resp = ollama.embed(model=EMBED_MODEL, input=f"search_query: {text}")
    return resp["embeddings"][0]


def hybrid_scale(dense, sparse, alpha=0.7):
    return (
        [v * alpha for v in dense],
        {"indices": sparse["indices"], "values": [v * (1 - alpha) for v in sparse["values"]]},
    )


def fetch_parents(parent_ids: list[str]) -> list[dict]:
    if not parent_ids:
        return []
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur  = conn.cursor()
    cur.execute("""
        SELECT parent_id, question_number, full_text, total_marks,
               course_code, semester, year
        FROM   parent_chunks
        WHERE  parent_id = ANY(%s)
    """, (parent_ids,))
    rows = cur.fetchall()
    cur.close(); conn.close()
    return [
        {
            "parent_id":       r[0],
            "question_number": r[1],
            "full_text":       r[2],
            "total_marks":     r[3],
            "course_code":     r[4],
            "semester":        r[5],
            "year":            r[6],
        }
        for r in rows
    ]


def retrieve(query, course_code=None, top_k=20, rerank_top_n=5, alpha=0.7):
    dense  = embed_query(query)
    sparse = bm25.encode_queries([query])[0]
    d, s   = hybrid_scale(dense, sparse, alpha)

    pinecone_filter = {"course_code": {"$eq": course_code}} if course_code else None

    results = index.query(
        vector=d, sparse_vector=s, top_k=top_k,
        include_metadata=True, filter=pinecone_filter,
    )
    if not results.matches:
        return []

    passages = [m.metadata.get("text_preview", "") for m in results.matches]
    scores   = reranker.predict([(query, p) for p in passages])
    ranked   = sorted(zip(results.matches, scores), key=lambda x: x[1], reverse=True)
    top      = [m for m, _ in ranked[:rerank_top_n]]

    parent_ids = list({m.metadata["parent_id"] for m in top})
    return fetch_parents(parent_ids)


def generate_answer(query, parents):
    if not parents:
        return "No relevant questions found."
    parts = []
    for doc in parents:
        header = (
            f"[{doc['course_code']} | {doc['semester']} {doc['year']} | "
            f"Q{doc['question_number']} | {doc['total_marks']} marks]"
        )
        parts.append(f"{header}\n{doc['full_text']}")
    context = "\n\n---\n\n".join(parts)
    prompt  = f"EXAM QUESTIONS:\n{context}\n\nSTUDENT: {query}"
    resp = gen_model.generate_content(prompt, generation_config={"temperature": 0.1})
    return resp.text

print("✅ Section 1-2 dependencies re-initialized")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Section 1-2 dependencies re-initialized


In [11]:
# ── 6.0 DeepEval judge setup — Gemini ─────────────────────────────────────────
import os, json, time, statistics
from pydantic import BaseModel
from typing import Optional
import google.generativeai as genai
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase
from dotenv import load_dotenv
load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))


class GeminiDeepEval(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-3.1-flash-lite"):
        self.model_name = model_name
        self.model = genai.GenerativeModel(model_name)

    def load_model(self):
        return self.model

    def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> str:
        full_prompt = prompt
        if schema is not None:
            full_prompt = (
                f"{prompt}\n\n"
                f"Respond ONLY with a valid JSON object matching this schema:\n"
                f"{json.dumps(schema.model_json_schema(), indent=2)}\n"
                f"No markdown, no explanation, no preamble. "
                f"Ensure the JSON is complete and properly closed. "
                f"Start your response with {{"
            )

        response = self.model.generate_content(
            full_prompt,
            generation_config={"temperature": 0, "max_output_tokens": 2000},
        )
        raw = response.text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()

        if schema is not None:
            start = raw.find("{")
            end   = raw.rfind("}")
            if start != -1 and end != -1:
                raw = raw[start:end+1]

        return raw

    async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> str:
        return self.generate(prompt, schema)

    def get_model_name(self) -> str:
        return self.model_name


os.environ["DEEPEVAL_EMBEDDING_MODEL"] = "models/gemini-embedding-2-preview"
os.environ["GOOGLE_API_KEY"]           = os.getenv("GEMINI_API_KEY")

gemini_eval = GeminiDeepEval()

faithfulness_metric = FaithfulnessMetric(threshold=0.5, model=gemini_eval, include_reason=True)
answer_rel_metric   = AnswerRelevancyMetric(threshold=0.5, model=gemini_eval, include_reason=True)

print("✅ DeepEval ready — Gemini 2.0 Flash judge + Gemini embeddings")

✅ DeepEval ready — Gemini 2.0 Flash judge + Gemini embeddings


In [12]:
# ── 6.1 Sample queries for RAGAS ──────────────────────────────────────────────
import random
random.seed(42)

RAGAS_SAMPLE_SIZE = 40

with open("data/eval_dataset_final.json") as f:
    eval_dataset = json.load(f)

topic_items = [item for item in eval_dataset if item["query_type"] == "topic_search"]

by_course = {}
for item in topic_items:
    c = item["course_code"] or "unknown"
    by_course.setdefault(c, []).append(item)

sampled = []
per_course = max(1, RAGAS_SAMPLE_SIZE // len(by_course))
for course, items in by_course.items():
    sampled.extend(random.sample(items, min(per_course, len(items))))

remaining = [i for i in topic_items if i not in sampled]
random.shuffle(remaining)
sampled.extend(remaining[:max(0, RAGAS_SAMPLE_SIZE - len(sampled))])
sampled = sampled[:RAGAS_SAMPLE_SIZE]

print(f"✅ Sampled {len(sampled)} queries across {len(by_course)} courses")
for course, items in by_course.items():
    n = sum(1 for s in sampled if (s["course_code"] or "unknown") == course)
    print(f"  {course}: {n} sampled / {len(items)} available")

✅ Sampled 40 queries across 2 courses
  EDB2613: 20 sampled / 25 available
  RBB3013: 20 sampled / 20 available


In [14]:
# ── 6.2 Build test cases ──────────────────────────────────────────────────────
test_cases = []

def generate_answer_with_retry(query, parents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return generate_answer(query, parents)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"       ⚠️  Gemini error ({type(e).__name__}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"       ❌ Failed after {max_retries} attempts: {e}")
                return "ERROR: generation failed"

print(f"Building test cases for {len(sampled)} queries...")
print()

for i, item in enumerate(sampled):
    query  = item["query"]
    course = item["course_code"]

    print(f"[{i+1:02d}/{len(sampled)}] {query[:60]}")

    try:
            parents = retrieve(query, course_code=course, rerank_top_n=5, alpha=0.7)
    except Exception as e:
            print(f"       ⚠️  retrieve() failed: {type(e).__name__}: {e}")
            continue
    if not parents:
        print("       ⚠️  No results — skipping")
        continue

    answer = generate_answer_with_retry(query, parents)
    if answer.startswith("ERROR:"):
        continue

    contexts = [p["full_text"] for p in parents]

    test_cases.append(LLMTestCase(
        input             = query,
        actual_output     = answer,
        retrieval_context = contexts,
    ))

    time.sleep(0.3)

print(f"✅ {len(test_cases)} usable test cases after filtering")

Building test cases for 40 queries...

[01/40] how to calibrate differential pressure transmitter for level
[02/40] how to evaluate transmitter accuracy and precision
[03/40] how to find transfer function poles and zeros
[04/40] how to calculate excitation voltage for wheatstone bridge st
[05/40] how to select a thermocouple for high temperature measuremen
[06/40] how to use Routh-Hurwitz criterion for system stability
[07/40] how to calibrate differential pressure level transmitter
[08/40] how to find transfer function from overshoot and settling ti
[09/40] how to calculate steady state error for control systems
[10/40] how to calculate average and standard deviation of measureme
[11/40] can a P controller achieve zero steady state offset
[12/40] how to calculate LRV and URV for level measurement
[13/40] how to design transfer function from step response parameter
[14/40] flow control loop primary elements and square root character
[15/40] how to calculate measurement sensitivity from

In [15]:
# ── 6.3 Sanity check on ONE test case (run BEFORE the full batch) ─────────────
tc_test = test_cases[0]

try:
    faithfulness_metric.measure(tc_test)
    print(f"Faithfulness: {faithfulness_metric.score}")
    print(f"Reason: {faithfulness_metric.reason}")
except Exception as e:
    print(f"❌ Faithfulness failed: {e}")

try:
    answer_rel_metric.measure(tc_test)
    print(f"Relevancy: {answer_rel_metric.score}")
    print(f"Reason: {answer_rel_metric.reason}")
except Exception as e:
    print(f"❌ Relevancy failed: {e}")

Output()

Output()

Faithfulness: 1.0
Reason: The score is 1.00 because the actual output is perfectly aligned with the retrieval context, demonstrating excellent accuracy!


Relevancy: 1.0
Reason: The score is 1.00 because the response provided a perfectly accurate and relevant guide to calibrating a differential pressure transmitter for level measurement with no extraneous information.


In [19]:
# ── 6.4 Batch scoring — incremental save after each test case ─────────────────
import traceback
from pathlib import Path

SCORED_PATH = Path("data/scored_progress.json")

# Resume if file exists
if SCORED_PATH.exists():
    with open(SCORED_PATH) as f:
        scored = json.load(f)
    done_queries = {s["query"] for s in scored}
    print(f"Resuming — {len(scored)} already done")
else:
    scored = []
    done_queries = set()
    print("Starting fresh")

print(f"🚀 Scoring {len(test_cases)} test cases individually...")
print()

for i, tc in enumerate(test_cases):
    if tc.input in done_queries:
        continue

    entry = {
        "query":            tc.input,
        "faithfulness":     None,
        "faithfulness_why": None,
        "answer_relevancy": None,
        "relevancy_why":    None,
    }

    print(f"[{i+1:02d}/{len(test_cases)}] {tc.input[:60]}")

    for attempt in range(2):
        try:
            faithfulness_metric.measure(tc)
            entry["faithfulness"]     = faithfulness_metric.score
            entry["faithfulness_why"] = faithfulness_metric.reason
            break
        except Exception as e:
            err_str = str(e)
            print(f"       ⚠️  Faithfulness error: {type(e).__name__}: {err_str[:300]}")
            if "429" in err_str or "rate" in err_str.lower() or "quota" in err_str.lower():
                print("       🚨 RATE LIMIT — sleeping 30s")
                time.sleep(60)
            elif attempt == 0:
                time.sleep(2)
            else:
                traceback.print_exc()

    for attempt in range(2):
        try:
            answer_rel_metric.measure(tc)
            entry["answer_relevancy"] = answer_rel_metric.score
            entry["relevancy_why"]    = answer_rel_metric.reason
            break
        except Exception as e:
            err_str = str(e)
            print(f"       ⚠️  Relevancy error: {type(e).__name__}: {err_str[:300]}")
            if "429" in err_str or "rate" in err_str.lower() or "quota" in err_str.lower():
                print("       🚨 RATE LIMIT — sleeping " \
                "60s")
                time.sleep(60)
            elif attempt == 0:
                time.sleep(2)
            else:
                traceback.print_exc()

    scored.append(entry)
    done_queries.add(tc.input)

    # Save after EVERY test case — survives kernel crash, not just interrupt
    with open(SCORED_PATH, "w") as f:
        json.dump(scored, f, indent=2)

    time.sleep(1)

print("\n✅ Batch scoring complete!")

Output()

Resuming — 12 already done
🚀 Scoring 40 test cases individually...

[13/40] how to design transfer function from step response parameter


Output()

Output()

[14/40] flow control loop primary elements and square root character


...

Output()

Output()

[40/40] advantages and disadvantages of PI controllers in control sy


Output()


✅ Batch scoring complete!


In [20]:
# ── 6.5 Summary ────────────────────────────────────────────────────────────────
valid = [s for s in scored if s["faithfulness"] is not None and s["answer_relevancy"] is not None]

if valid:
    mean_faithfulness     = statistics.mean(s["faithfulness"]     for s in valid)
    mean_answer_relevancy = statistics.mean(s["answer_relevancy"] for s in valid)
else:
    mean_faithfulness     = 0.0
    mean_answer_relevancy = 0.0
    print("⚠️  No valid samples — check judge model errors above")

print(f"Mean Faithfulness    : {mean_faithfulness:.3f}")
print(f"Mean Answer Relevancy: {mean_answer_relevancy:.3f}")
print(f"Valid samples        : {len(valid)}/{len(scored)}")
print()

print(f"{'Query':<50} {'Faith':>6} {'Relev':>6}")
print("-" * 65)
for s in scored:
    f = f"{s['faithfulness']:.2f}"     if s["faithfulness"]     is not None else "  err"
    r = f"{s['answer_relevancy']:.2f}" if s["answer_relevancy"] is not None else "  err"
    print(f"{s['query'][:49]:<50} {f:>6} {r:>6}")

Mean Faithfulness    : 1.000
Mean Answer Relevancy: 0.518
Valid samples        : 36/36

Query                                               Faith  Relev
-----------------------------------------------------------------
how to calibrate differential pressure transmitte    1.00   1.00
how to evaluate transmitter accuracy and precisio    1.00   0.00
how to find transfer function poles and zeros        1.00   0.20
how to calculate excitation voltage for wheatston    1.00   0.62
how to select a thermocouple for high temperature    1.00   0.19
how to use Routh-Hurwitz criterion for system sta    1.00   0.41
how to calibrate differential pressure level tran    1.00   0.33
how to find transfer function from overshoot and     1.00   0.19
how to calculate steady state error for control s    1.00   0.40
how to calculate average and standard deviation o    1.00   0.00
can a P controller achieve zero steady state offs    1.00   1.00
how to calculate LRV and URV for level measuremen    1.00   0.00
h

---
## Section 7 — Final Summary & Export

In [29]:
print("=" * 60)
print("PAPERSLOTH — RAG EVALUATION SUMMARY")
print("=" * 60)
print()
print("  RETRIEVAL QUALITY (Precision@5)")
print(f"    Hybrid dense-sparse (α=0.7) : {mean_hybrid:.3f}")
print(f"    Dense-only (α=1.0)          : {mean_dense:.3f}")
print(f"    Improvement                 : {improvement:+.1f}%")
print()
print("  GENERATION QUALITY (DeepEval)")
print(f"    Faithfulness                : {mean_faithfulness:.3f}")
print(f"    Answer Relevancy            : {mean_answer_relevancy:.3f}")
print()
print("  DATASET")
print(f"    Total eval queries          : {len(eval_dataset)}")
print(f"    Topic queries               : {len(topic_results)}")
print(f"    Negative queries            : {len(negative_results)}")
print(f"    DeepEval samples            : {len(valid)}")
print("=" * 60)

PAPERSLOTH — RAG EVALUATION SUMMARY

  RETRIEVAL QUALITY (Precision@5)
    Hybrid dense-sparse (α=0.7) : 0.241
    Dense-only (α=1.0)          : 0.237
    Improvement                 : +1.9%

  GENERATION QUALITY (DeepEval)
    Faithfulness                : 1.000
    Answer Relevancy            : 0.518

  DATASET
    Total eval queries          : 55
    Topic queries               : 45
    Negative queries            : 10
    DeepEval samples            : 36


In [30]:
export = {
    "summary": {
        "precision_at_5_hybrid": mean_hybrid,
        "precision_at_5_dense":  mean_dense,
        "improvement_pct":       improvement,
        "faithfulness":          mean_faithfulness,
        "answer_relevancy":      mean_answer_relevancy,
        "n_eval_queries":        len(eval_dataset),
        "n_deepeval_samples":    len(valid),
    },
    "per_query_precision": ablation_results,
    "per_query_generation": scored,
}

from pathlib import Path
Path("data").mkdir(exist_ok=True)
with open("data/evaluation_results.json", "w") as f:
    json.dump(export, f, indent=2)

print("✅ Results saved → data/evaluation_results.json")

✅ Results saved → data/evaluation_results.json
